# Stage 2 — Adaptive Synthetic Allocation & DiffusionBoost

This notebook is the **single entry point** for Stage 2: migrations, CIFAR-100 SD generation (optional flag), **global FID** (clean-fid), Tiny ImageNet full grid, reduced CIFAR-100 track, and figures.

Training uses **synthetic-aware weighted CE** when `synthetic_aware_loss: true` in YAML and a same-architecture baseline checkpoint is passed (notebook does this automatically). Evaluation writes **temperature scaling**, **linear probe** (sklearn on frozen features), **covariance eigen-spectrum** (+ `eval_eigenvalues.png`), and **linear CKA vs baseline** (+ `eval_cka_heatmap.png`) into each run’s `metrics.json`.

**Prerequisites:** Stage 1 data (`data/tiny_imagenet_5pct/train_index.csv`, raw Tiny ImageNet). Synthetic pool: `data/synthetic_sd/` or `data/synthetic/tiny_imagenet/` (migration links legacy layout).

Use **Run All** after editing flags in the next cell (`RUN_CIFAR_GENERATE` / `RUN_FID` are heavy).

In [1]:
# ---- Run configuration ----
import os, sys, json
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

# Master switches (edit before Run All) — all True = full end-to-end pipeline (long: CIFAR SD + FID + full Tiny grid)
RUN_MIGRATION = True          # symlink legacy synthetic_sd → canonical layout; link Stage 1 checkpoints
RUN_CIFAR_GENERATE = True     # generate CIFAR-100 synthetic cache if missing (very long SD run)
RUN_FID = True                # global FID Tiny: 5% real vs synthetic at each ratio (needs pip install clean-fid)
RUN_TINY_EXPERIMENTS = True   # full Tiny ImageNet training grid
RUN_CIFAR_EXPERIMENTS = True  # reduced CIFAR-100 (R18): baseline, uniform 15x, utility, adaptive, ceiling
RUN_FIGURES = True            # plot summary charts from results/ into figures/stage2/

# Tiny: set False to skip heavy sections during debugging
TINY_TRAIN_BASELINES = True
TINY_TRAIN_UNIFORM_SCALING = True   # ResNet-18 only: 5x, 10x, 15x
TINY_TRAIN_UNIFORM_15X_BOTH_ARCH = True
TINY_TRAIN_ADAPTIVE = True          # uses allocation CSVs (hard_class, uncertainty, predicted_utility)
TINY_TRAIN_CEILING = True

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Project:", PROJECT_ROOT)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

Project: /mnt/data/cv
CUDA: True NVIDIA GeForce RTX 4060 Ti


## I. Environment & migrations
Resolves `data/synthetic_sd` into `data/synthetic/tiny_imagenet` (symlink or redirect file on Windows) and links `checkpoints/` under `results/tiny_imagenet/legacy/`.

In [2]:
from src.stage2.orchestrator import Stage2Orchestrator

orch = Stage2Orchestrator(PROJECT_ROOT)

if RUN_MIGRATION:
    synth_path = orch.migrate_tiny_synthetic()
    print("Synthetic (Tiny) resolved to:", synth_path)
    orch.link_stage1_checkpoints()
    print("Legacy checkpoints linked under results/.../legacy/")

if RUN_CIFAR_GENERATE:
    orch.ensure_cifar100_synthetic(force=False)
    print("CIFAR-100 synthetic ready under data/synthetic/cifar100")

if RUN_FID:
    fid_tiny = orch.compute_global_fid("tiny_imagenet.yaml", ratios=[5, 10, 15])
    print("Tiny ImageNet global FID:", fid_tiny)
    fid_cifar = orch.compute_global_fid("cifar100.yaml", ratios=[15])
    print("CIFAR-100 global FID (15x budget):", fid_cifar)

Synthetic (Tiny) resolved to: /mnt/data/cv/data/synthetic_sd
Legacy checkpoints linked under results/.../legacy/
CIFAR-100 synthetic ready under data/synthetic/cifar100
compute FID between two folders
Found 5000 images in the folder /mnt/data/cv/results/tiny_imagenet/fid_cache/real_5pct_png


FID real_5pct_png : 100%|██████████| 157/157 [00:46<00:00,  3.41it/s]


Found 25000 images in the folder /mnt/data/cv/results/tiny_imagenet/fid_cache/synth_uniform_5x_125pc


FID synth_uniform_5x_125pc : 100%|██████████| 782/782 [08:12<00:00,  1.59it/s]


compute FID between two folders
Found 5000 images in the folder /mnt/data/cv/results/tiny_imagenet/fid_cache/real_5pct_png


...

Tiny ImageNet global FID: {'fid_5x': 140.04724977469834, 'fid_10x': 139.60714257869586, 'fid_15x': 139.47715362344053}
Files already downloaded and verified
Files already downloaded and verified
compute FID between two folders
Found 2500 images in the folder /mnt/data/cv/results/cifar100/fid_cache/real_5pct_png


FID real_5pct_png : 100%|██████████| 79/79 [00:24<00:00,  3.21it/s]


Found 37260 images in the folder /mnt/data/cv/results/cifar100/fid_cache/synth_uniform_15x_375pc


FID synth_uniform_15x_375pc : 100%|██████████| 1165/1165 [02:34<00:00,  7.55it/s]


CIFAR-100 global FID (15x budget): {'fid_15x': 76.6385727349471}


## II. Tiny ImageNet — training & evaluation
Order: baselines (R18 + MobileNetV3-Small) → uniform scaling (R18) → uniform 15× (both) → diagnostics & allocations → adaptive (per policy) → ceiling.
Metrics and checkpoints are written to `results/tiny_imagenet/{pipeline}/{arch}/{timestamp}/`.

In [3]:
if not RUN_TINY_EXPERIMENTS:
    print("Skipping Tiny experiments.")
else:
    baseline_runs = {}
    uniform15 = {}

    if TINY_TRAIN_BASELINES:
        for arch in ["resnet18", "mobilenet_v3_small"]:
            rd, m = orch.train_baseline("tiny_imagenet.yaml", arch)
            baseline_runs[arch] = {"run_dir": str(rd), "metrics": m}
            print(arch, "baseline top1", m["top1"])

    r18_ckpt = Path(baseline_runs["resnet18"]["run_dir"]) / "best.pt" if baseline_runs.get("resnet18") else None

    def _tiny_ckpt(a):
        br = baseline_runs.get(a)
        return Path(br["run_dir"]) / "best.pt" if br else None

    if TINY_TRAIN_UNIFORM_SCALING and baseline_runs:
        ck = _tiny_ckpt("resnet18")
        for ratio in [5, 10, 15]:
            rd, m = orch.train_uniform(
                "tiny_imagenet.yaml", "resnet18", ratio, baseline_ckpt_same_arch=ck
            )
            print("uniform", ratio, "x top1", m["top1"])

    if TINY_TRAIN_UNIFORM_15X_BOTH_ARCH and baseline_runs:
        for arch in ["resnet18", "mobilenet_v3_small"]:
            ck = _tiny_ckpt(arch)
            rd, m = orch.train_uniform(
                "tiny_imagenet.yaml", arch, 15, baseline_ckpt_same_arch=ck
            )
            uniform15[arch] = {"run_dir": str(rd), "metrics": m}
            print(arch, "uniform 15x top1", m["top1"])

    utility_path = PROJECT_ROOT / "results" / "tiny_imagenet" / "utility_from_uniform15x.json"
    if uniform15.get("resnet18") and r18_ckpt and r18_ckpt.is_file():
        from src.data.registry import class_ids_in_label_order, get_baseline_loaders
        from src.data.transforms import get_train_transform, get_val_transform
        from src.config import load_experiment_config

        cfg = load_experiment_config(orch.config_path("tiny_imagenet.yaml"), PROJECT_ROOT)
        tr_t = get_train_transform(cfg.dataset.image_size)
        va_t = get_val_transform(cfg.dataset.image_size)
        _, _, c2i = get_baseline_loaders(cfg, tr_t, va_t)
        cids = class_ids_in_label_order(c2i)
        mb = baseline_runs["resnet18"]["metrics"]
        mu = uniform15["resnet18"]["metrics"]
        util = orch.utility_from_metrics(mb, mu, cids)
        utility_path.parent.mkdir(parents=True, exist_ok=True)
        utility_path.write_text(json.dumps(util, indent=2), encoding="utf-8")
        print("Saved utility targets to", utility_path)

        diag_path = orch.compute_baseline_diagnostics(
            "tiny_imagenet.yaml", r18_ckpt, arch="resnet18", quality_csv=None
        )
        print("Diagnostics CSV:", diag_path)

        alloc_dir = orch.build_allocations(
            "tiny_imagenet.yaml",
            diag_path,
            utility_path if utility_path.exists() else None,
            policies=["hard_class", "uncertainty", "predicted_utility"],
        )
        print("Allocation CSVs:", alloc_dir)

    if TINY_TRAIN_ADAPTIVE and baseline_runs:
        alloc_root = PROJECT_ROOT / "results" / "tiny_imagenet" / "allocations"
        for pol in ["hard_class", "uncertainty", "predicted_utility"]:
            csvp = alloc_root / f"allocation_{pol}.csv"
            if csvp.is_file():
                for arch in ["resnet18", "mobilenet_v3_small"]:
                    ck = _tiny_ckpt(arch)
                    rd, m = orch.train_adaptive(
                        "tiny_imagenet.yaml",
                        arch,
                        csvp,
                        name=f"adaptive_15x_{pol}",
                        baseline_ckpt_same_arch=ck,
                    )
                    print(arch, pol, "top1", m["top1"])

    if TINY_TRAIN_CEILING and baseline_runs:
        for arch in ["resnet18", "mobilenet_v3_small"]:
            ck = _tiny_ckpt(arch)
            rd, m = orch.train_ceiling(
                "tiny_imagenet.yaml", arch, baseline_ckpt_same_arch=ck
            )
            print(arch, "ceiling top1", m["top1"])

    orch.aggregate_results_index("tiny_imagenet.yaml")
    print("Tiny ImageNet results index updated.")

/mnt/data/cv/src/training/stage2_train.py:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=mixed_precision)
Train:   0%|          | 0/79 [00:00<?, ?it/s]/mnt/data/cv/src/training/train_eval.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/30  train_loss=4.8968 acc=0.0598  val_loss=4.1221 acc=0.1383


Epoch 2/30  train_loss=3.8343 acc=0.1902  val_loss=3.3615 acc=0.2563


Epoch 3/30  train_loss=3.2430 acc=0.3024  val_loss=3.0443 acc=0.3136


Epoch 4/30  train_loss=2.8118 acc=0.3858  val_loss=2.9210 acc=0.3360


...

Epoch 28/30  train_loss=0.7277 acc=0.8276  val_loss=2.2099 acc=0.5014


Epoch 29/30  train_loss=0.7151 acc=0.8260  val_loss=2.2073 acc=0.5012


/mnt/data/cv/src/stage2/orchestrator.py:169: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(run_dir / "best.pt", map_location=self.device)


Epoch 30/30  train_loss=0.7676 acc=0.8184  val_loss=2.2096 acc=0.5020
Early stopping at epoch 30


AttributeError: 'MobileNetV3' object has no attribute 'fc'

## II-b. Resume Tiny ImageNet after crash

In [4]:
# Reload the fixed backbone module so forward_features picks up the fix
import importlib, src.models.backbone, src.stage2.diagnostics, src.evaluation.stage2_eval
importlib.reload(src.models.backbone)
importlib.reload(src.stage2.diagnostics)
importlib.reload(src.evaluation.stage2_eval)

# Skip if Cell 5 already succeeded (baseline_runs populated for both archs)
_need_resume = "baseline_runs" not in dir() or len(globals().get("baseline_runs", {})) < 2

if _need_resume and RUN_TINY_EXPERIMENTS:
    print("Resuming Tiny ImageNet experiments from disk ...")

    # Discover latest completed run per pipeline/arch from results/tiny_imagenet/
    def _find_latest_run(pipeline: str, arch: str) -> Path | None:
        parent = PROJECT_ROOT / "results" / "tiny_imagenet" / pipeline / arch
        if not parent.is_dir():
            return None
        candidates = sorted(
            [d for d in parent.iterdir() if d.is_dir() and (d / "best.pt").is_file()],
            key=lambda d: d.name,
        )
        return candidates[-1] if candidates else None

    def _load_metrics(run_dir: Path) -> dict:
        mp = run_dir / "metrics.json"
        if mp.is_file():
            return json.loads(mp.read_text(encoding="utf-8"))
        return {}

    # --- Baselines ---
    baseline_runs = {}
    for arch in ["resnet18", "mobilenet_v3_small"]:
        rd = _find_latest_run("baseline", arch)
        m = _load_metrics(rd) if rd else {}
        if rd and m:
            baseline_runs[arch] = {"run_dir": str(rd), "metrics": m}
            print(f"  {arch} baseline recovered: top1={m.get('top1')}")
        else:
            print(f"  {arch} baseline not found on disk, retraining ...")
            rd, m = orch.train_baseline("tiny_imagenet.yaml", arch)
            baseline_runs[arch] = {"run_dir": str(rd), "metrics": m}
            print(f"  {arch} baseline top1={m['top1']}")

    r18_ckpt = Path(baseline_runs["resnet18"]["run_dir"]) / "best.pt" if baseline_runs.get("resnet18") else None

    def _tiny_ckpt(a):
        br = baseline_runs.get(a)
        return Path(br["run_dir"]) / "best.pt" if br else None

    # --- Re-evaluate any baseline that trained but crashed before eval ---
    for arch in ["resnet18", "mobilenet_v3_small"]:
        br = baseline_runs.get(arch)
        if br and not br["metrics"]:
            print(f"  {arch} baseline: best.pt exists but no metrics — re-evaluating ...")
            from src.models.backbone import build_backbone
            from src.evaluation.stage2_eval import evaluate_stage2
            from src.data.registry import get_baseline_loaders
            from src.data.transforms import get_train_transform, get_val_transform
            from src.config import load_experiment_config

            cfg = load_experiment_config(orch.config_path("tiny_imagenet.yaml"), PROJECT_ROOT)
            tr_t = get_train_transform(cfg.dataset.image_size)
            va_t = get_val_transform(cfg.dataset.image_size)
            _, val_loader, c2i = get_baseline_loaders(cfg, tr_t, va_t)
            model = build_backbone(arch, cfg.dataset.num_classes)
            rd_path = Path(br["run_dir"])
            model.load_state_dict(torch.load(rd_path / "best.pt", map_location="cpu"))
            model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
            m = evaluate_stage2(
                model, val_loader, cfg, c2i,
                torch.device("cuda" if torch.cuda.is_available() else "cpu"),
                run_dir=rd_path,
            )
            baseline_runs[arch]["metrics"] = m
            print(f"  {arch} baseline top1={m['top1']}")

    # --- Uniform scaling (R18 5x/10x/15x) ---
    if TINY_TRAIN_UNIFORM_SCALING and baseline_runs:
        ck = _tiny_ckpt("resnet18")
        for ratio in [5, 10, 15]:
            rd, m = orch.train_uniform(
                "tiny_imagenet.yaml", "resnet18", ratio, baseline_ckpt_same_arch=ck
            )
            print("uniform", ratio, "x top1", m["top1"])

    # --- Uniform 15x both archs ---
    uniform15 = {}
    if TINY_TRAIN_UNIFORM_15X_BOTH_ARCH and baseline_runs:
        for arch in ["resnet18", "mobilenet_v3_small"]:
            ck = _tiny_ckpt(arch)
            rd, m = orch.train_uniform(
                "tiny_imagenet.yaml", arch, 15, baseline_ckpt_same_arch=ck
            )
            uniform15[arch] = {"run_dir": str(rd), "metrics": m}
            print(arch, "uniform 15x top1", m["top1"])

    # --- Utility + diagnostics + allocations ---
    utility_path = PROJECT_ROOT / "results" / "tiny_imagenet" / "utility_from_uniform15x.json"
    if uniform15.get("resnet18") and r18_ckpt and r18_ckpt.is_file():
        from src.data.registry import class_ids_in_label_order, get_baseline_loaders
        from src.data.transforms import get_train_transform, get_val_transform
        from src.config import load_experiment_config

        cfg = load_experiment_config(orch.config_path("tiny_imagenet.yaml"), PROJECT_ROOT)
        tr_t = get_train_transform(cfg.dataset.image_size)
        va_t = get_val_transform(cfg.dataset.image_size)
        _, _, c2i = get_baseline_loaders(cfg, tr_t, va_t)
        cids = class_ids_in_label_order(c2i)
        mb = baseline_runs["resnet18"]["metrics"]
        mu = uniform15["resnet18"]["metrics"]
        util = orch.utility_from_metrics(mb, mu, cids)
        utility_path.parent.mkdir(parents=True, exist_ok=True)
        utility_path.write_text(json.dumps(util, indent=2), encoding="utf-8")
        print("Saved utility targets to", utility_path)

        diag_path = orch.compute_baseline_diagnostics(
            "tiny_imagenet.yaml", r18_ckpt, arch="resnet18", quality_csv=None
        )
        print("Diagnostics CSV:", diag_path)

        alloc_dir = orch.build_allocations(
            "tiny_imagenet.yaml",
            diag_path,
            utility_path if utility_path.exists() else None,
            policies=["hard_class", "uncertainty", "predicted_utility"],
        )
        print("Allocation CSVs:", alloc_dir)

    # --- Adaptive ---
    if TINY_TRAIN_ADAPTIVE and baseline_runs:
        alloc_root = PROJECT_ROOT / "results" / "tiny_imagenet" / "allocations"
        for pol in ["hard_class", "uncertainty", "predicted_utility"]:
            csvp = alloc_root / f"allocation_{pol}.csv"
            if csvp.is_file():
                for arch in ["resnet18", "mobilenet_v3_small"]:
                    ck = _tiny_ckpt(arch)
                    rd, m = orch.train_adaptive(
                        "tiny_imagenet.yaml",
                        arch,
                        csvp,
                        name=f"adaptive_15x_{pol}",
                        baseline_ckpt_same_arch=ck,
                    )
                    print(arch, pol, "top1", m["top1"])

    # --- Ceiling ---
    if TINY_TRAIN_CEILING and baseline_runs:
        for arch in ["resnet18", "mobilenet_v3_small"]:
            ck = _tiny_ckpt(arch)
            rd, m = orch.train_ceiling(
                "tiny_imagenet.yaml", arch, baseline_ckpt_same_arch=ck
            )
            print(arch, "ceiling top1", m["top1"])

    orch.aggregate_results_index("tiny_imagenet.yaml")
    print("Tiny ImageNet results index updated.")
elif not _need_resume:
    print("Cell 5 already completed — nothing to resume.")

Resuming Tiny ImageNet experiments from disk ...
  resnet18 baseline recovered: top1=0.4559
  mobilenet_v3_small baseline not found on disk, retraining ...


Epoch 1/30  train_loss=5.0417 acc=0.0414  val_loss=4.3279 acc=0.1347


Epoch 2/30  train_loss=3.7122 acc=0.1962  val_loss=3.0880 acc=0.2885


Epoch 3/30  train_loss=2.8216 acc=0.3374  val_loss=2.7422 acc=0.3480


Epoch 4/30  train_loss=2.3519 acc=0.4276  val_loss=2.4968 acc=0.3988


...

Epoch 27/30  train_loss=0.7309 acc=0.8286  val_loss=2.1632 acc=0.5107


Epoch 28/30  train_loss=0.7214 acc=0.8300  val_loss=2.1652 acc=0.5111


Epoch 29/30  train_loss=0.7755 acc=0.8174  val_loss=2.1639 acc=0.5113


Epoch 30/30  train_loss=0.7475 acc=0.8184  val_loss=2.1651 acc=0.5120


AttributeError: 'MobileNetV3' object has no attribute 'fc'

## III. CIFAR-100 (reduced generalisation track)
ResNet-18 only: **baseline** → **uniform 15×** (synthetic-aware CE + eval vs baseline CKA) → **per-class utility** → allocations (**hard_class** + **predicted_utility**) → **adaptive 15×** using `scope.cifar_adaptive_policy` in `configs/cifar100.yaml` (default **predicted_utility**) → **ceiling**.

In [ ]:
if not RUN_CIFAR_EXPERIMENTS:
    print("Skipping CIFAR experiments.")
else:
    rdir, mb = orch.train_baseline("cifar100.yaml", "resnet18")
    ck = Path(rdir) / "best.pt"
    _, m_uni = orch.train_uniform(
        "cifar100.yaml", "resnet18", 15, baseline_ckpt_same_arch=ck
    )
    from src.data.registry import class_ids_in_label_order, get_baseline_loaders
    from src.data.transforms import get_train_transform, get_val_transform
    from src.config import load_experiment_config

    cfg_c = load_experiment_config(orch.config_path("cifar100.yaml"), PROJECT_ROOT)
    tr_t = get_train_transform(cfg_c.dataset.image_size)
    va_t = get_val_transform(cfg_c.dataset.image_size)
    _, _, c2i = get_baseline_loaders(cfg_c, tr_t, va_t)
    cids = class_ids_in_label_order(c2i)
    util_c = orch.utility_from_metrics(mb, m_uni, cids)
    utility_path_c = PROJECT_ROOT / "results" / "cifar100" / "utility_from_uniform15x.json"
    utility_path_c.parent.mkdir(parents=True, exist_ok=True)
    utility_path_c.write_text(json.dumps(util_c, indent=2), encoding="utf-8")
    print("Saved CIFAR utility targets to", utility_path_c)

    diag_c = orch.compute_baseline_diagnostics("cifar100.yaml", ck, arch="resnet18")
    orch.build_allocations(
        "cifar100.yaml",
        diag_c,
        utility_path_c,
        policies=["hard_class", "predicted_utility"],
    )
    pol = cfg_c.scope.cifar_adaptive_policy
    acsv = PROJECT_ROOT / "results" / "cifar100" / "allocations" / f"allocation_{pol}.csv"
    if acsv.is_file():
        orch.train_adaptive(
            "cifar100.yaml",
            "resnet18",
            acsv,
            name=f"adaptive_15x_{pol}",
            baseline_ckpt_same_arch=ck,
        )
    else:
        print("Missing allocation CSV:", acsv)
    orch.train_ceiling("cifar100.yaml", "resnet18", baseline_ckpt_same_arch=ck)
    orch.aggregate_results_index("cifar100.yaml")
    print("CIFAR-100 reduced track complete.")

## IV. Figures (from saved results)
Reads `results/*/results_index.json` and writes quick comparison plots to `figures/stage2/`.

In [ ]:
if RUN_FIGURES:
    import matplotlib.pyplot as plt
    cfg_t = orch.load_cfg("tiny_imagenet.yaml")
    cfg_t.path_figures.mkdir(parents=True, exist_ok=True)
    idx_path = cfg_t.path_results_root / "tiny_imagenet" / "results_index.json"
    if idx_path.is_file():
        rows = json.loads(idx_path.read_text(encoding="utf-8"))
        # Last run per pipeline/arch
        best = {}
        for r in rows:
            key = (r["pipeline"], r["arch"])
            best[key] = r
        names = [f"{a[0]}\n{a[1]}" for a in sorted(best.keys())][:12]
        vals = [best[k]["top1"] for k in sorted(best.keys())][:12]
        plt.figure(figsize=(10, 4))
        plt.bar(range(len(names)), vals, color="steelblue")
        plt.xticks(range(len(names)), names, rotation=45, ha="right", fontsize=9)
        plt.ylabel("Val top-1")
        plt.tight_layout()
        outp = cfg_t.path_figures / "summary_top1_last_runs.png"
        plt.savefig(outp, dpi=300)
        plt.savefig(outp.with_suffix(".pdf"))
        plt.close()
        print("Saved", outp)
    else:
        print("No results index yet; run training cells first.")

## V. Submission checklist
- `results/` contains per-run `metrics.json`, `best.pt`, `training_curves.json`.
- Figures under `figures/stage2/` (PNG + PDF).
- Export Stage 2 report PDF with numbers filled from `metrics.json` / `results_index.json`.